In [49]:
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Check if GPU is available and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


Transform the images into 256x256 pixels, for a consistent input size for the model.

In [50]:
import os
from PIL import Image

# Define the input and output directories
input_dir = "data"
output_dir = "data_resized"
model_dir = "models"

if not os.path.exists(model_dir):
    os.makedirs(model_dir, exist_ok=True)

if os.path.exists(output_dir):
    print(f"Output directory '{output_dir}' already exists. Please remove it or choose a different name.")
else:
    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Loop through each folder and image in the input directory
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith((".png", ".jpg", ".jpeg")):  # Check for image files
                input_path = os.path.join(root, file)
                
                # Open the image and resize it
                with Image.open(input_path) as img:
                    img_resized = img.resize((256, 256))
                    
                    # Create the corresponding output path
                    relative_path = os.path.relpath(root, input_dir)
                    output_folder = os.path.join(output_dir, relative_path)
                    os.makedirs(output_folder, exist_ok=True)
                    output_path = os.path.join(output_folder, file)
                    
                    # Save the resized image
                    img_resized.save(output_path)

    print("Images have been resized and saved to 'data_resized'.")

Output directory 'data_resized' already exists. Please remove it or choose a different name.


In [51]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiTaskCNN(nn.Module):
    def __init__(self):
        super(MultiTaskCNN, self).__init__()

        # Input: 256x256 → after 3 MaxPool(2,2): 256 / 2^3 = 32
        input_size = 256
        num_pools = 3
        pooled_size = input_size // (2 ** num_pools)          # 32
        flattened_size = pooled_size * pooled_size * 64        # 32*32*64 = 65536

        # Convolutional layers
        self.conv1 = nn.Conv2d(in_channels=3,  out_channels=16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)

        # Pooling layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully connected layers for multi-class classification (0-4 index)
        self.fc1 = nn.Linear(flattened_size, 128)
        self.fc2 = nn.Linear(128, 5)

    def forward(self, x):
        # Convolutional layers with ReLU and pooling
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = self.pool(x)

        # Flatten the output for fully connected layers
        x = x.view(x.size(0), -1)

        # Multi-class classification branch (0-4 severity levels)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)  # Raw logits for CrossEntropyLoss

        return x

# Instantiate and display
if 'device' not in locals():
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = MultiTaskCNN().to(device)
print(model)

MultiTaskCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=65536, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=5, bias=True)
)


We now load the resized images and their corresponding labels into memory. 
The label is the name of the folder that contains the image, which represents the class of the image.

In [52]:
from typing import Optional

class ResizedImageDataset(Dataset):
    def __init__(self, root_dir: str, transform: Optional[transforms.Compose] = None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths: list[str] = []
        self.labels: list[str] = []

        # Loop through the directory to collect image paths and labels
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file.endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(root, file))
                    self.labels.append(os.path.basename(root))  # Folder name as label

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx: int):
        img_path = self.image_paths[idx]
        label: str = self.labels[idx]
        
        # Open the image
        with Image.open(img_path) as img:
            if self.transform:
                img = self.transform(img)
        
        if not isinstance(img, torch.Tensor):
            img = transforms.ToTensor()(img)  # Ensure the image is a tensor

        return img, label

# Define transformations for the dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    # transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Normalize to [-1, 1]
])

# Create the dataset
dataset = ResizedImageDataset(output_dir, transform=transform)

# Example: Load the first image and label
img, label = dataset[random.randint(0, len(dataset)-1)]
print(f"Image shape: {img.shape}, Label: {label}")

Image shape: torch.Size([3, 256, 256]), Label: 2-moderate


In [53]:
# ==================== Configuration ====================
num_epochs = 100
batch_size = 32
learning_rate = 0.001
weight_decay = 1e-5
model_save_interval = 20  # Save model every 20 epochs
model_save_path = os.path.join(model_dir, "multi_task_cnn.pth")

# ==================== Data Loading ====================
# Create DataLoader for training and validation
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# ==================== Training Setup ====================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)  # Reduce LR by 50% every 10 epochs

# ==================== Training Loop ====================
model.train()
model.to(device) # just in case

for epoch in range(num_epochs):
    total_loss = 0.0
    
    for images, labels in train_loader:
        # Move data to device
        images = images.to(device)
        index_labels = torch.tensor([int(label.split('-')[0]) for label in labels], dtype=torch.long).to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, index_labels)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    # Update learning rate
    scheduler.step()
    
    # Print progress
    avg_loss = total_loss / len(train_loader)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, LR: {current_lr:.6f}")
    
    # Save model every model_save_interval epochs
    if (epoch + 1) % model_save_interval == 0:
        torch.save(model.state_dict(), model_save_path)
        print(f"Model saved to {model_save_path}")

print("Training completed!")

Epoch [1/100], Loss: 1.4977, LR: 0.001000
Epoch [2/100], Loss: 1.4218, LR: 0.001000
Epoch [3/100], Loss: 1.4231, LR: 0.001000
Epoch [4/100], Loss: 1.4081, LR: 0.001000
Epoch [5/100], Loss: 1.4047, LR: 0.001000
Epoch [6/100], Loss: 1.3844, LR: 0.001000
Epoch [7/100], Loss: 1.3849, LR: 0.001000
Epoch [8/100], Loss: 1.3561, LR: 0.001000
Epoch [9/100], Loss: 1.3587, LR: 0.001000
Epoch [10/100], Loss: 1.3658, LR: 0.000500
Epoch [11/100], Loss: 1.3207, LR: 0.000500
Epoch [12/100], Loss: 1.3026, LR: 0.000500
Epoch [13/100], Loss: 1.2707, LR: 0.000500
Epoch [14/100], Loss: 1.2341, LR: 0.000500
Epoch [15/100], Loss: 1.2232, LR: 0.000500
Epoch [16/100], Loss: 1.1791, LR: 0.000500
Epoch [17/100], Loss: 1.1863, LR: 0.000500
Epoch [18/100], Loss: 1.1295, LR: 0.000500
Epoch [19/100], Loss: 1.0907, LR: 0.000500
Epoch [20/100], Loss: 1.0405, LR: 0.000250
Model saved to models/multi_task_cnn.pth
Epoch [21/100], Loss: 0.9714, LR: 0.000250
Epoch [22/100], Loss: 0.9114, LR: 0.000250
Epoch [23/100], Loss: 

In [54]:
# Save the model's state dictionary
model_save_path = os.path.join(model_dir, "multi_task_cnn_final.pth")
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

Model saved to models/multi_task_cnn_final.pth


In [57]:
from sklearn.metrics import accuracy_score

model_save_path = os.path.join(model_dir, "multi_task_cnn_final.pth")
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# Load the model's state dictionary
model.load_state_dict(torch.load(model_save_path, map_location=device))
model.to(device)  # Move model to the device

# Initialize lists to store predictions and true labels
preds = []
true_labels = []

# Set the model to evaluation mode
model.eval()

# Perform validation
with torch.no_grad():
    for images, labels in val_loader:
        # Move data to the device
        images = images.to(device)
        index_labels = torch.tensor([int(label.split('-')[0]) for label in labels], dtype=torch.long).to(device)

        # Forward pass
        out = model(images)

        # Collect predictions
        preds.extend(torch.argmax(out, dim=1).cpu().numpy())
        true_labels.extend(index_labels.cpu().numpy())

# Calculate accuracy for multi-class classification
accuracy = accuracy_score(true_labels, preds)

print(f"Validation Accuracy: {accuracy:.4f}")

Validation Accuracy: 0.3689


In [56]:
import random
import os

# Get a random image and its label from the dataset
random_idx = random.randint(0, len(dataset) - 1)
img, label = dataset[random_idx]

# Move the image to the device and add a batch dimension
img = img.unsqueeze(0).to(device)

# Set the model to evaluation mode
model.eval()

# Perform inference
with torch.no_grad():
    output = model(img)
    predicted_class = torch.argmax(output, dim=1).item()

# Map the predicted class index to the folder name
class_mapping = {0: "0-noDR", 1: "1-mild", 2: "2-moderate", 3: "3-severe", 4: "4-proliferativeDR"}
predicted_label = class_mapping[predicted_class]

# Print the results
print(f"True Label: {label}, Predicted Label: {predicted_label}")

# Verify if the image is in the correct folder
image_path = dataset.image_paths[random_idx]
print(f"Image Path: {image_path}")
print(f"Is the image in the correct folder? {'Yes' if predicted_label in image_path else 'No'}")

True Label: 1-mild, Predicted Label: 1-mild
Image Path: data_resized/1-mild/20621_left.jpeg
Is the image in the correct folder? Yes
